# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# dataset.metadata is a dataclass, get its fields for inspection
print("Dataset Metadata Overview:")
print(f"Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Published Date: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (all referenced by their `@id` values).

Let's list all record sets, and for each, show their fields with IDs and descriptions. This allows us to select fields to process in later steps.

In [ ]:
# List all record sets, with all internal IDs for recordset and fields
print("Available Record Sets and Fields:")
record_sets = dataset.metadata.recordSet  # This should be a list of RecordSet objects
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id_}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    record_set_ids.append(rs.id_)
    if hasattr(rs, 'field') and rs.field:
        print("  Fields:")
        for field in rs.field:
            print(f"    - Field @id: {field.id_}")
            print(f"      Name: {getattr(field, 'name', '')}")
            print(f"      DataType: {getattr(field, 'dataType', '')}")
            print(f"      Description: {getattr(field, 'description', '')}")
    else:
        print("  No fields found for this record set.")
    print()
if not record_set_ids:
    print("No record sets declared in the schema. Please inspect the dataset documentation for source files or try loading records directly.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

We'll attempt to load all record sets. If the dataset has only one, use that.

In [ ]:
# If record_set_ids wasn't populated previously due to potential schema irregularities, try querying dynamic record set list
if not record_sets or not record_set_ids:
    # fallback: try the dataset interface
    print("Attempting dynamic discovery of recordSet IDs...")
    from mlcroissant.dataset.dataset import Dataset
    # This is a hack for known Croissant v1.0 datasets that list distributions as record sets
    if hasattr(dataset.metadata, 'distribution') and dataset.metadata.distribution:
        record_set_ids = [d.id_ for d in dataset.metadata.distribution]
        print(f"Distribution IDs as record sets: {record_set_ids}")
    else:
        print("No available record sets.")

dataframes = {}
successfully_loaded = []
for rec_id in record_set_ids:
    try:
        # Some datasets require the full '@id' string, others just identifier
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records from record set: {rec_id}")
        print(f"Columns: {df.columns.tolist()}")
        successfully_loaded.append(rec_id)
    except Exception as e:
        print(f"Failed to load from record set {rec_id}: {e}")

# Set the primary record set we'll use for later steps
if successfully_loaded:
    primary_record_set = successfully_loaded[0]  # Pick the first as default
    print(f"\nPrimary record set: {primary_record_set}")
    print(dataframes[primary_record_set].head())
else:
    print("No record set data could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (filtering, normalizing, grouping) on the main record set. All column references below should correspond to their field `@id`s.

In [ ]:
import numpy as np
if successfully_loaded:
    df = dataframes[primary_record_set].copy()
    # Show all columns
    print("All available columns in this record set:")
    print(df.columns.tolist())

    # Attempt to pick a numeric field for analysis - we'll heuristically use first float/integer column
    potential_numeric = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[c])]
    print(f"Potential numeric columns: {potential_numeric}")
    if potential_numeric:
        numeric_field = potential_numeric[0]
    else:
        # Try to force convert a column named 'MW' or 'Coverage' or similar to float
        fallback_candidates = [c for c in df.columns if 'MW' in c or 'coverage' in c.lower() or 'abundance' in c.lower()]
        for col in fallback_candidates:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_field = col
                    break
            except Exception:
                pass
        else:
            print("Could not identify a numeric field for filtering.")
            numeric_field = None

    if numeric_field is not None:
        # Filter: show records where value > threshold
        threshold = df[numeric_field].quantile(0.9)  # Use top 10% as a threshold example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field if available (e.g., 'description' or any object col with < 10 unique vals)
        group_candidate = None
        for col in df.columns:
            if col == numeric_field:
                continue
            if df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < 10:
                group_candidate = col
                break
        if group_candidate:
            grouped_df = filtered_df.groupby(group_candidate)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_candidate} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric field to demonstrate EDA steps.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if successfully_loaded and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped by group_candidate field
    if 'grouped_df' in locals() and group_candidate:
        plt.figure(figsize=(8, 4))
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f'Mean {numeric_field} by {group_candidate}')
        plt.xlabel(group_candidate)
        plt.ylabel(f'Mean {numeric_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded comprehensive metadata from the FAIR² Croissant package for the Mass Spectrometry protein dataset.
- Explored available record sets and fields as defined by their `@id`.
- Loaded tabular data for analysis and demonstrated common EDA steps (filter, normalization, grouping) with all column operations performed by `@id`.
- Visualized numeric field distributions and simple grouped summaries.

For in-depth analysis, consult the [FAIR² dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and use the schema's `@id` values for any programmatic access.
